In [2]:
import os
import glob
from langchain_community.document_loaders import DirectoryLoader, PDFMinerLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma

C:\Users\IB200200\AppData\Local\Temp\ipykernel_20724\4152558772.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, PDFMinerLoader


### Data preparation

In [3]:
folders = glob.glob("my-knowledge/*")

documents = []
for folder in folders:
    category = os.path.basename(folder)
    loader = DirectoryLoader(folder, glob="**/*.pdf", loader_cls=PDFMinerLoader)
    folder_docs = loader.load()
    for doc in folder_docs:
        doc.metadata["category"] = category
        doc.metadata["name"] = doc.metadata["source"].split("\\")[-1][:-4]
        documents.append(doc)

In [5]:
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

Created a chunk of size 1009, which is longer than the specified 1000
Created a chunk of size 1019, which is longer than the specified 1000
Created a chunk of size 1048, which is longer than the specified 1000
Created a chunk of size 1443, which is longer than the specified 1000
Created a chunk of size 1082, which is longer than the specified 1000
Created a chunk of size 1009, which is longer than the specified 1000
Created a chunk of size 1218, which is longer than the specified 1000
Created a chunk of size 1231, which is longer than the specified 1000
Created a chunk of size 1411, which is longer than the specified 1000
Created a chunk of size 1046, which is longer than the specified 1000
Created a chunk of size 1392, which is longer than the specified 1000
Created a chunk of size 1487, which is longer than the specified 1000
Created a chunk of size 1523, which is longer than the specified 1000
Created a chunk of size 1391, which is longer than the specified 1000
Created a chunk of s

### Vector database creation

In [7]:
embedding_model = "jeffh/intfloat-multilingual-e5-large-instruct:f32"
db_name = "fit_db"

In [8]:
embeddings = OllamaEmbeddings(model=embedding_model)

In [ ]:
# if exist to clear

if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

In [9]:
# create vectorstore

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")

Vectorstore created with 323 documents
